# Selecting, filtering and transforming

## Learning objectives

By the end of this notebook you will be able to:

- select columns, and rows by label with loc and by position with iloc
- filter rows with conditions, isin, between and query
- sort by one or more columns
- create columns with vectorised maths, np.where, cut, map and apply
- work with dates: extract parts, compute intervals, resample

## Introduction

Most analysis code selects some rows and columns, adds a calculated variable, sorts the result and
looks at it. This notebook covers those everyday operations, using the patients and lab results tables
and a simulated two-year sales series.

In [1]:
import pandas as pd
import numpy as np
RAW = "../datasets/raw/"
PROCESSED = "../datasets/processed/"

In [2]:
patients = pd.read_csv(RAW + "patients.csv", parse_dates=["enrolled"])
labs = pd.read_csv(RAW + "lab_results.csv", parse_dates=["visit_date"])
patients.head()

,patient_id,site,sex,age,weight_kg,height_cm,smoker,group,enrolled
0,P0001,Site A,M,41,66.4,156.3,No,Treatment,2025-03-31
1,P0002,Site B,M,83,69.5,168.0,No,Control,2025-08-19
2,P0003,Site B,F,36,71.4,172.2,No,Control,2025-08-11
3,P0004,Site A,F,24,66.0,158.9,Yes,Control,2025-06-24
4,P0005,Site A,M,77,80.5,163.7,No,Control,2025-08-05


## Selecting columns

In [3]:
patients[["patient_id", "age", "sex"]].head(3)

,patient_id,age,sex
0,P0001,41,M
1,P0002,83,M
2,P0003,36,F


In [4]:
patients.filter(like="_").head(3)          # columns whose name contains "_"

,patient_id,weight_kg,height_cm
0,P0001,66.4,156.3
1,P0002,69.5,168.0
2,P0003,71.4,172.2


In [5]:
patients.select_dtypes("number").head(3)

,age,weight_kg,height_cm
0,41,66.4,156.3
1,83,69.5,168.0
2,36,71.4,172.2


## loc and iloc

- `df.loc[rows, columns]` selects by **label** (index labels and column names); slices **include** the end
- `df.iloc[rows, columns]` selects by **position**; slices exclude the end, like Python

In [6]:
print(patients.iloc[0, 3])                  # first row, fourth column
patients.iloc[:3, :4]

41


,patient_id,site,sex,age
0,P0001,Site A,M,41
1,P0002,Site B,M,83
2,P0003,Site B,F,36


In [7]:
patients.loc[0:2, ["patient_id", "age"]]   # labels 0, 1 and 2

,patient_id,age
0,P0001,41
1,P0002,83
2,P0003,36


In [8]:
p = patients.set_index("patient_id")
p.loc[["P0003", "P0010"], ["site", "age", "group"]]

,site,age,group
patient_id,,,
P0003,Site B,36,Control
P0010,Site A,40,Control


## Filtering rows

A condition gives a True/False Series; using it inside `[]` or `.loc[]` keeps the True rows. As with
NumPy, combine with `&`, `|`, `~` and brackets.

In [9]:
older = patients[patients["age"] >= 65]
print(len(older), "patients aged 65 or more")
patients[(patients["sex"] == "F") & (patients["smoker"] == "Yes")].head()

93 patients aged 65 or more


,patient_id,site,sex,age,weight_kg,height_cm,smoker,group,enrolled
3,P0004,Site A,F,24,66.0,158.9,Yes,Control,2025-06-24
26,P0027,Site A,F,20,79.4,163.3,Yes,Treatment,2025-04-06
30,P0031,Site A,F,76,69.9,177.0,Yes,Control,2025-11-09
37,P0038,Site B,F,18,100.5,169.8,Yes,Control,2025-04-10
40,P0041,Site C,F,77,85.8,163.7,Yes,Treatment,2025-02-09


In [10]:
patients[patients["site"].isin(["Site A", "Site C"])].shape

(165, 9)

In [11]:
patients[patients["age"].between(30, 39)].shape

(44, 9)

`query` writes the same conditions as a string, which is easier to read for long filters. `@` refers
to a Python variable:

In [12]:
min_age = 60
patients.query("group == 'Treatment' and age >= @min_age and smoker == 'No'").head()

,patient_id,site,sex,age,weight_kg,height_cm,smoker,group,enrolled
15,P0016,Site C,F,67,84.5,160.0,No,Treatment,2025-11-07
18,P0019,Site A,F,77,65.4,179.0,No,Treatment,2025-11-26
21,P0022,Site B,F,75,81.8,168.5,No,Treatment,2025-05-26
24,P0025,Site B,M,73,81.9,179.8,No,Treatment,2025-01-11
25,P0026,Site A,M,81,68.4,146.5,No,Treatment,2025-07-29


Filtering and assigning at once uses `loc` with a condition and a column:

In [13]:
p2 = patients.copy()
p2.loc[p2["weight_kg"].isna(), "weight_flag"] = "missing"
p2["weight_flag"].value_counts(dropna=False)

weight_flag
NaN        290
missing     10
Name: count, dtype: int64

## Sorting

In [14]:
patients.sort_values("age", ascending=False).head(3)

,patient_id,site,sex,age,weight_kg,height_cm,smoker,group,enrolled
66,P0067,Site A,M,85,57.8,160.4,No,Control,2025-11-15
131,P0132,Site C,F,85,81.1,165.1,No,Treatment,2025-02-18
125,P0126,Site A,M,85,64.4,174.4,No,Control,2025-01-17


In [15]:
patients.sort_values(["site", "age"], ascending=[True, False]).head(4)

,patient_id,site,sex,age,weight_kg,height_cm,smoker,group,enrolled
66,P0067,Site A,M,85,57.8,160.4,No,Control,2025-11-15
125,P0126,Site A,M,85,64.4,174.4,No,Control,2025-01-17
155,P0156,Site A,F,84,84.1,174.6,Yes,Control,2025-09-10
167,P0168,Site A,F,84,38.0,157.5,Yes,Control,2025-11-30


In [16]:
patients.nlargest(3, "weight_kg")[["patient_id", "weight_kg"]]

,patient_id,weight_kg
297,P0298,101.6
37,P0038,100.5
132,P0133,98.8


## Creating new columns

### Vectorised arithmetic

The fastest and clearest way: operate on whole columns.

In [17]:
patients["height_m"] = patients["height_cm"] / 100
patients["bmi"] = (patients["weight_kg"] / patients["height_m"] ** 2).round(1)
patients[["weight_kg", "height_cm", "bmi"]].head()

,weight_kg,height_cm,bmi
0,66.4,156.3,27.2
1,69.5,168.0,24.6
2,71.4,172.2,24.1
3,66.0,158.9,26.1
4,80.5,163.7,30.0


`assign` creates columns and returns a new DataFrame, which chains well:

In [18]:
(patients
 .assign(age_decade=lambda d: d["age"] // 10 * 10,
         is_smoker=lambda d: d["smoker"] == "Yes")
 [["patient_id", "age", "age_decade", "is_smoker"]]
 .head())

,patient_id,age,age_decade,is_smoker
0,P0001,41,40,False
1,P0002,83,80,False
2,P0003,36,30,False
3,P0004,24,20,True
4,P0005,77,70,False


### Conditions: np.where and np.select

In [19]:
patients["older"] = np.where(patients["age"] >= 65, "65+", "under 65")
patients["bmi_class"] = np.select(
    [patients["bmi"] < 18.5, patients["bmi"] < 25, patients["bmi"] < 30, patients["bmi"] >= 30],
    ["Underweight", "Normal", "Overweight", "Obese"], default="Unknown")
patients["bmi_class"].value_counts()

bmi_class
Normal         116
Overweight      80
Obese           68
Underweight     26
Unknown         10
Name: count, dtype: int64

`default="Unknown"` catches the rows where BMI is missing, because NaN fails every comparison.

### Binning: cut

In [20]:
patients["age_group"] = pd.cut(patients["age"], bins=[17, 34, 49, 64, 120],
                               labels=["18-34", "35-49", "50-64", "65+"])
patients["age_group"].value_counts().sort_index()

age_group
18-34    79
35-49    65
50-64    63
65+      93
Name: count, dtype: int64

`bins` are edges; each bin includes its right edge, so 34 falls in 18-34. `pd.qcut` makes bins with
equal numbers of rows instead (quartiles, deciles).

### map: translate values

In [21]:
patients["sex_label"] = patients["sex"].map({"F": "Female", "M": "Male"})
patients["sex_label"].value_counts()

sex_label
Female    153
Male      147
Name: count, dtype: int64

### apply: any function

`apply` runs a Python function on each value (Series) or each row (`axis=1`). It is flexible but slow
on large data; prefer vectorised operations when one exists.

In [22]:
def risk_label(row):
    if row["age"] >= 65 and row["smoker"] == "Yes":
        return "high"
    if row["age"] >= 65 or row["smoker"] == "Yes":
        return "moderate"
    return "low"

patients["risk"] = patients.apply(risk_label, axis=1)
patients["risk"].value_counts()

risk
low         162
moderate    114
high         24
Name: count, dtype: int64

The same result, vectorised. The two methods agree:

In [23]:
n_flags = (patients["age"] >= 65).astype(int) + (patients["smoker"] == "Yes").astype(int)
vectorised = n_flags.map({0: "low", 1: "moderate", 2: "high"})
print((vectorised == patients["risk"]).all())

True


## String columns

The `.str` accessor applies string methods to every value:

In [24]:
print(patients["patient_id"].str[1:].astype(int).head(3).tolist())
print(patients["site"].str.replace("Site ", "").head(3).tolist())
print(patients["site"].str.contains("A").sum())

[1, 2, 3]
['A', 'B', 'B']
111


## Dates

The `.dt` accessor gives parts of dates.

In [25]:
labs["year"] = labs["visit_date"].dt.year
labs["month"] = labs["visit_date"].dt.month_name()
labs["weekday"] = labs["visit_date"].dt.day_name()
labs[["visit_date", "year", "month", "weekday"]].head()

,visit_date,year,month,weekday
0,2025-04-02,2025,April,Wednesday
1,2025-05-01,2025,May,Thursday
2,2025-08-19,2025,August,Tuesday
3,2025-09-19,2025,September,Friday
4,2025-10-17,2025,October,Friday


Differences between dates are `Timedelta` values; `.dt.days` turns them into numbers:

In [26]:
first_visit = labs.groupby("patient_id")["visit_date"].transform("min")
labs["days_since_first"] = (labs["visit_date"] - first_visit).dt.days
labs[["patient_id", "visit", "visit_date", "days_since_first"]].head(6)

,patient_id,visit,visit_date,days_since_first
0,P0001,0,2025-04-02,0
1,P0001,1,2025-05-01,29
2,P0002,0,2025-08-19,0
3,P0002,1,2025-09-19,31
4,P0002,2,2025-10-17,59
5,P0003,0,2025-08-08,0


### Time series: resampling

The sales data have one row per shop per day. `resample` regroups by time period; it needs the dates
as the index.

In [27]:
sales = pd.read_csv(RAW + "daily_sales.csv", parse_dates=["date"])
sales["revenue"] = sales["units"] * sales["price"]
north = sales[sales["shop"] == "North"].set_index("date")
monthly = north["revenue"].resample("MS").sum().round(0)
monthly.head()

date
2024-01-01    13100.0
2024-02-01    13659.0
2024-03-01    16748.0
2024-04-01    19110.0
2024-05-01    21060.0
Freq: MS, Name: revenue, dtype: float64

`"MS"` means month start. Others: `"D"` day, `"W"` week, `"QS"` quarter, `"YS"` year. A rolling mean
smooths day-to-day noise:

In [28]:
north["units_7day"] = north["units"].rolling(7).mean()
north[["units", "units_7day"]].iloc[5:10]

,units,units_7day
date,,
2024-01-06,128,NaN
2024-01-07,84,98.714286
2024-01-08,121,102.285714
2024-01-09,81,97.000000
2024-01-10,75,95.428571


The first six rows of a 7-day rolling mean are NaN because there are not yet seven days to average.

## Method chaining

Several steps written as one readable pipeline:

In [29]:
(patients
 .query("smoker == 'Yes'")
 .assign(bmi_rounded=lambda d: d["bmi"].round())
 .sort_values("bmi", ascending=False)
 [["patient_id", "site", "age", "bmi"]]
 .head(5))

,patient_id,site,age,bmi
112,P0113,Site C,42,40.9
53,P0054,Site B,74,39.6
240,P0241,Site C,24,38.0
170,P0171,Site A,25,37.5
297,P0298,Site A,81,35.2


## Practise

Exercises for this folder are in `exercises/Pandas_Exercises.ipynb`.

## Further reading

- [pandas user guide](https://pandas.pydata.org/docs/user_guide/index.html)
- [Indexing and selecting data](https://pandas.pydata.org/docs/user_guide/indexing.html)
- [Time series and date functionality](https://pandas.pydata.org/docs/user_guide/timeseries.html)
- [Copy-on-write in pandas](https://pandas.pydata.org/docs/user_guide/copy_on_write.html)